In [ ]:
```json
{
  "nbformat": 4,
  "nbformat_minor": 5,
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "codemirror_mode": {
        "name": "python",
        "version": 3
      },
      "file_extension": ".py",
      "mimetype": "text/x-python",
      "name": "python",
      "nbconvert_exporter": "python",
      "pygments_lexer": "ipython3",
      "version": "3.10.12"
    },
    "custom_metadata": {
      "databricks": {
        "lang": "sql",
        "clusters": [
          {
            "id": "0000-000000-cluster"
          }
        ],
        "workspace_path": "/Users/user@example.com/WC_BADGE_PRODUCT_D_Load"
      }
    }
  },
  "cells": [
    {
      "cell_type": "markdown",
      "source": [
        "# WC_BADGE_PRODUCT_D_Load\n",
        "\n",
        "**Conversion Timestamp**: 2024-07-30 12:00:00\n",
        "\n",
        "This notebook migrates the ODI session for `WC_BADGE_PRODUCT_D` to Databricks Spark SQL. It handles staging, flow, and merge logic, applying necessary data type and syntax conversions, and ensuring Delta Lake compatibility."
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "dbutils.widgets.text(\"ETL_JOB_TYPE\", \"\", \"1. ETL Job Type\")\n",
        "dbutils.widgets.text(\"DATASOURCE_NUM_ID\", \"\", \"2. Datasource Number ID\")\n",
        "dbutils.widgets.text(\"ETLProcWID\", \"\", \"3. ETL Process WID\")\n",
        "dbutils.widgets.text(\"ODI_SESS_NO\", \"\", \"4. ODI Session Number\")"
      ],
      "metadata": {
        "language": "python"
      },
      "outputs": []
    },
    {
      "cell_type": "markdown",
      "source": [
        "## ETL Parameters"
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "-- MAGIC %sql\n",
        "CREATE OR REPLACE TEMPORARY VIEW v_etl_last_extract_time AS\n",
        "SELECT etl_last_extract_time AS etl_last_extract_time\n",
        "FROM workspace.prxbi_dw.wc_etl_parameters\n",
        "WHERE ETL_JOB_TYPE = '${ETL_JOB_TYPE}'; -- SCEN_TASK_NO {1}, {3}\n",
        "\n",
        "CREATE OR REPLACE TEMPORARY VIEW v_etl_current_extract_time AS\n",
        "SELECT etl_current_extract_time AS etl_current_extract_time\n",
        "FROM workspace.prxbi_dw.wc_etl_parameters\n",
        "WHERE ETL_JOB_TYPE = '${ETL_JOB_TYPE}'; -- SCEN_TASK_NO {2}, {4}\n",
        "\n",
        "CREATE OR REPLACE TEMPORARY VIEW v_row_wid_param AS\n",
        "SELECT ROW_WID AS row_wid_param\n",
        "FROM workspace.prxbi_dw.wc_etl_parameters\n",
        "WHERE ETL_JOB_TYPE = '${ETL_JOB_TYPE}'; -- SCEN_TASK_NO {5}, {6}"
      ],
      "metadata": {
        "language": "sql"
      },
      "outputs": []
    },
    {
      "cell_type": "code",
      "source": [
        "display(spark.sql(\"\"\"\n",
        "SELECT\n",
        "  '${ETL_JOB_TYPE}' AS ETL_JOB_TYPE,\n",
        "  ${DATASOURCE_NUM_ID} AS DATASOURCE_NUM_ID,\n",
        "  ${ETLProcWID} AS ETL_PROC_WID,\n",
        "  '${ODI_SESS_NO}' AS ODI_SESS_NO,\n",
        "  (SELECT etl_last_extract_time FROM v_etl_last_extract_time) AS ETL_LAST_EXTRACT_TIME,\n",
        "  (SELECT etl_current_extract_time FROM v_etl_current_extract_time) AS ETL_CURRENT_EXTRACT_TIME,\n",
        "  (SELECT row_wid_param FROM v_row_wid_param) AS ROW_WID_PARAM\n",
        "\"\"\"))"
      ],
      "metadata": {
        "language": "python"
      },
      "outputs": []
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Staging Table: `C$_0FILTER`"
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {30}: Drop staging table\n",
        "DROP TABLE IF EXISTS workspace.prxbi_dw.c_0filter;"
      ],
      "metadata": {
        "language": "sql"
      },
      "outputs": []
    },
    {
      "cell_type": "code",
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {40}: Create staging table\n",
        "CREATE TABLE workspace.prxbi_dw.c_0filter\n",
        "(\n",
        "  NAME STRING,\n",
        "  PRICE DECIMAL(25,5),\n",
        "  STATUS BIGINT,\n",
        "  VISIBILITY BIGINT,\n",
        "  WEIGHT BIGINT,\n",
        "  ID STRING,\n",
        "  PARENT_SKU STRING,\n",
        "  PARENT_RX_EBS_PRODUCT_CODE STRING,\n",
        "  CREATED_AT TIMESTAMP,\n",
        "  UPDATED_AT TIMESTAMP,\n",
        "  STOCK_QTY BIGINT,\n",
        "  RX_LINK_TYPE STRING\n",
        ")\n",
        "USING DELTA;"
      ],
      "metadata": {
        "language": "sql"
      },
      "outputs": []
    },
    {
      "cell_type": "code",
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {50}, {60}, {70}: Insert into staging table\n",
        "INSERT INTO workspace.prxbi_dw.c_0filter (\n",
        "  NAME,\n",
        "  PRICE,\n",
        "  STATUS,\n",
        "  VISIBILITY,\n",
        "  WEIGHT,\n",
        "  ID,\n",
        "  PARENT_SKU,\n",
        "  PARENT_RX_EBS_PRODUCT_CODE,\n",
        "  CREATED_AT,\n",
        "  UPDATED_AT,\n",
        "  STOCK_QTY,\n",
        "  RX_LINK_TYPE\n",
        ")\n",
        "SELECT\n",
        "  DISTINCT_PRODUCT_D.NAME AS NAME ,\n",
        "  DISTINCT_PRODUCT_D.PRICE AS PRICE ,\n",
        "  DISTINCT_PRODUCT_D.STATUS AS STATUS ,\n",
        "  DISTINCT_PRODUCT_D.VISIBILITY AS VISIBILITY ,\n",
        "  DISTINCT_PRODUCT_D.WEIGHT AS WEIGHT ,\n",
        "  DISTINCT_PRODUCT_D.ID AS ID ,\n",
        "  DISTINCT_PRODUCT_D.PARENT_SKU AS PARENT_SKU ,\
        "  DISTINCT_PRODUCT_D.PARENT_RX_EBS_PRODUCT_CODE AS PARENT_RX_EBS_PRODUCT_CODE ,\n",
        "  DISTINCT_PRODUCT_D.CREATED_AT AS CREATED_AT ,\n",
        "  DISTINCT_PRODUCT_D.UPDATED_AT AS UPDATED_AT ,\n",
        "  DISTINCT_PRODUCT_D.STOCK_QTY AS STOCK_QTY ,\n",
        "  DISTINCT_PRODUCT_D.RX_LINK_TYPE AS RX_LINK_TYPE\n",
        "FROM\n",
        "  (\n",
        "    SELECT\n",
        "      WC_MERCURY_PRODUCT_TS.NAME AS NAME,\n",
        "      WC_MERCURY_PRODUCT_TS.PARENT_PRICE AS PRICE,\n",
        "      WC_MERCURY_PRODUCT_TS.STATUS AS STATUS,\n",
        "      WC_MERCURY_PRODUCT_TS.VISIBILITY AS VISIBILITY,\n",
        "      WC_MERCURY_PRODUCT_TS.WEIGHT AS WEIGHT,\n",
        "      WC_MERCURY_PRODUCT_TS.INT_INSERT_DATE AS INT_INSERT_DATE,\n",
        "      WC_MERCURY_PRODUCT_TS.INT_UPDATE_DATE AS INT_UPDATE_DATE,\n",
        "      ROW_NUMBER() OVER(PARTITION BY WC_MERCURY_PRODUCT_TS.ID ORDER BY WC_MERCURY_PRODUCT_TS.INT_INSERT_DATE DESC) AS RNK,\n",
        "      WC_MERCURY_PRODUCT_TS.ID AS ID,\n",
        "      WC_MERCURY_PRODUCT_TS.SKU AS SKU,\n",
        "      WC_MERCURY_PRODUCT_TS.PARENT_SKU AS PARENT_SKU,\n",
        "      WC_MERCURY_PRODUCT_TS.PARENT_PRICE AS PARENT_PRICE,\n",
        "      WC_MERCURY_PRODUCT_TS.PARENT_RX_EBS_PRODUCT_CODE AS PARENT_RX_EBS_PRODUCT_CODE,\n",
        "      WC_MERCURY_PRODUCT_TS.CREATED_AT AS CREATED_AT,\n",
        "      WC_MERCURY_PRODUCT_TS.UPDATED_AT AS UPDATED_AT,\n",
        "      WC_MERCURY_PRODUCT_TS.STOCK_QTY AS STOCK_QTY,\n",
        "      WC_MERCURY_PRODUCT_TS.RX_LINK_TYPE AS RX_LINK_TYPE\n",
        "    FROM\n",
        "      workspace.prxbi_ts.wc_mercury_product_ts WC_MERCURY_PRODUCT_TS\n",
        "  ) AS DISTINCT_PRODUCT_D\n",
        "WHERE\n",
        "  (DISTINCT_PRODUCT_D.RNK = 1);"
      ],
      "metadata": {
        "language": "sql"
      },
      "outputs": []
    },
    {
      "cell_type": "code",
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {80}: Record count for staging table\n",
        "SELECT COUNT(*) FROM workspace.prxbi_dw.c_0filter;"
      ],
      "metadata": {
        "language": "sql"
      },
      "outputs": []
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Flow Table: `I$_WC_BADGE_PRODUCT_D`"
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {90}: Drop flow table\n",
        "DROP TABLE IF EXISTS workspace.prxbi_dw.i_wc_badge_product_d_flow;"
      ],
      "metadata": {
        "language": "sql"
      },
      "outputs": []
    },
    {
      "cell_type": "code",
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {100}: Create flow table\n",
        "CREATE TABLE workspace.prxbi_dw.i_wc_badge_product_d_flow\n",
        "(\n",
        "\tROW_WID BIGINT GENERATED BY DEFAULT AS IDENTITY,\n",
        "\tID STRING,\n",
        "\tSKU STRING,\n",
        "\tNAME STRING,\n",
        "\tPRICE DECIMAL(25,5),\n",
        "\tSTATUS BIGINT,\n",
        "\tVISIBILITY BIGINT,\n",
        "\tWEIGHT BIGINT,\n",
        "\tSTOCK_QTY BIGINT,\n",
        "\tINTEGRATION_ID STRING,\n",
        "\tDATASOURCE_NUM_ID STRING,\n",
        "\tW_INSERT_DT TIMESTAMP,\n",
        "\tW_UPDATE_DT TIMESTAMP,\n",
        "\tETL_PROC_WID BIGINT,\n",
        "\tPARENT_RX_EBS_PRODUCT_CODE STRING,\n",
        "\tCHANGED_ON_DT TIMESTAMP,\n",
        "\tCREATED_ON_DT TIMESTAMP,\n",
        "\tRX_LINK_TYPE STRING,\n",
        "\tIND_UPDATE STRING\n",
        ")\n",
        "USING DELTA;"
      ],
      "metadata": {
        "language": "sql"
      },
      "outputs": []
    },
    {
      "cell_type": "code",
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {110}: Insert into flow table\n",
        "INSERT INTO workspace.prxbi_dw.i_wc_badge_product_d_flow\n",
        "(\n",
        "\tID,\n",
        "\tSKU,\n",
        "\tNAME,\n",
        "\tPRICE,\n",
        "\tSTATUS,\n",
        "\tVISIBILITY,\n",
        "\tWEIGHT,\n",
        "\tSTOCK_QTY,\n",
        "\tINTEGRATION_ID,\n",
        "\tDATASOURCE_NUM_ID,\n",
        "\tW_INSERT_DT,\n",
        "\tW_UPDATE_DT,\n",
        "\tPARENT_RX_EBS_PRODUCT_CODE,\n",
        "\tCHANGED_ON_DT,\n",
        "\tCREATED_ON_DT,\n",
        "\tRX_LINK_TYPE,\n",
        "\tIND_UPDATE\n",
        ")\n",
        "SELECT\n",
        "\tS.ID,\n",
        "\tS.SKU,\n",
        "\tS.NAME,\n",
        "\tS.PRICE,\n",
        "\tS.STATUS,\n",
        "\tS.VISIBILITY,\n",
        "\tS.WEIGHT,\n",
        "\tS.STOCK_QTY,\n",
        "\tS.ID AS INTEGRATION_ID,\n",
        "\t'380' AS DATASOURCE_NUM_ID, -- Hardcoded as per ODI source\n",
        "\tcurrent_timestamp() AS W_INSERT_DT,\n",
        "\tcurrent_timestamp() AS W_UPDATE_DT,\n",
        "\tS.PARENT_RX_EBS_PRODUCT_CODE,\n",
        "\tS.UPDATED_AT AS CHANGED_ON_DT,\n",
        "\tS.CREATED_AT AS CREATED_ON_DT,\n",
        "\tS.RX_LINK_TYPE AS RX_LINK_TYPE,\n",
        "\t'I' AS IND_UPDATE\n",
        "FROM\n",
        "\tworkspace.prxbi_dw.c_0filter AS S\n",
        "WHERE NOT EXISTS\n",
        "\t( SELECT 1 FROM workspace.prxbi_dw.wc_badge_product_d AS T\n",
        "\tWHERE\n",
        "\t\tT.INTEGRATION_ID = S.ID AND T.DATASOURCE_NUM_ID = '380'\n",
        "\t\tAND ((T.ID = S.ID) OR (T.ID IS NULL AND S.ID IS NULL))\n",
        "\t\tAND ((T.SKU = S.PARENT_SKU) OR (T.SKU IS NULL AND S.PARENT_SKU IS NULL))\n",
        "\t\tAND ((T.NAME = S.NAME) OR (T.NAME IS NULL AND S.NAME IS NULL))\n",
        "\t\tAND ((T.PRICE = S.PRICE) OR (T.PRICE IS NULL AND S.PRICE IS NULL))\n",
        "\t\tAND ((T.STATUS = S.STATUS) OR (T.STATUS IS NULL AND S.STATUS IS NULL))\n",
        "\t\tAND ((T.VISIBILITY = S.VISIBILITY) OR (T.VISIBILITY IS NULL AND S.VISIBILITY IS NULL))\n",
        "\t\tAND ((T.WEIGHT = S.WEIGHT) OR (T.WEIGHT IS NULL AND S.WEIGHT IS NULL))\n",
        "\t\tAND ((T.STOCK_QTY = S.STOCK_QTY) OR (T.STOCK_QTY IS NULL AND S.STOCK_QTY IS NULL))\n",
        "\t\tAND ((T.W_UPDATE_DT = current_timestamp()) OR (T.W_UPDATE_DT IS NULL AND current_timestamp() IS NULL))\n",
        "\t\tAND ((T.PARENT_RX_EBS_PRODUCT_CODE = S.PARENT_RX_EBS_PRODUCT_CODE) OR (T.PARENT_RX_EBS_PRODUCT_CODE IS NULL AND S.PARENT_RX_EBS_PRODUCT_CODE IS NULL))\n",
        "\t\tAND ((T.CHANGED_ON_DT = S.UPDATED_AT) OR (T.CHANGED_ON_DT IS NULL AND S.UPDATED_AT IS NULL))\n",
        "\t\tAND ((T.CREATED_ON_DT = S.CREATED_AT) OR (T.CREATED_ON_DT IS NULL AND S.CREATED_AT IS NULL))\n",
        "\t\tAND ((T.RX_LINK_TYPE = S.RX_LINK_TYPE) OR (T.RX_LINK_TYPE IS NULL AND S.RX_LINK_TYPE IS NULL))\n",
        "\t);"
      ],
      "metadata": {
        "language": "sql"
      },
      "outputs": []
    },
    {
      "cell_type": "code",
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {120}, {130}: Optimize flow table (replaces create index and dbms_stats)\n",
        "-- Disable ZORDER stats check to prevent DELTA_ZORDERING_ON_COLUMN_WITHOUT_STATS\n",
        "SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;\n",
        "OPTIMIZE workspace.prxbi_dw.i_wc_badge_product_d_flow ZORDER BY (INTEGRATION_ID, DATASOURCE_NUM_ID);"
      ],
      "metadata": {
        "language": "sql"
      },
      "outputs": []
    },
    {
      "cell_type": "code",
      "source": [
        "-- MAGIC %sql\n",
        "-- Record count for flow table\n",
        "SELECT COUNT(*) FROM workspace.prxbi_dw.i_wc_badge_product_d_flow;"
      ],
      "metadata": {
        "language": "sql"
      },
      "outputs": []
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Mark Records for Update"
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {140}: Flag records for update (rewritten from tuple IN UPDATE to MERGE)\n",
        "MERGE INTO workspace.prxbi_dw.i_wc_badge_product_d_flow AS T\n",
        "USING (\n",
        "    SELECT INTEGRATION_ID, DATASOURCE_NUM_ID\n",
        "    FROM workspace.prxbi_dw.wc_badge_product_d\n",
        ") AS S\n",
        "ON T.INTEGRATION_ID = S.INTEGRATION_ID\n",
        "AND T.DATASOURCE_NUM_ID = S.DATASOURCE_NUM_ID\n",
        "WHEN MATCHED THEN UPDATE SET T.IND_UPDATE = 'U';"
      ],
      "metadata": {
        "language": "sql"
      },
      "outputs": []
    },
    {
      "cell_type": "markdown",
      "source": [
        "## MERGE into Target Table: `WC_BADGE_PRODUCT_D`"
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {150} (skipped), {160} (update), {170} (insert) combined into a single MERGE\n",
        "MERGE INTO workspace.prxbi_dw.wc_badge_product_d AS T\n",
        "USING workspace.prxbi_dw.i_wc_badge_product_d_flow AS S\n",
        "ON T.INTEGRATION_ID = S.INTEGRATION_ID\n",
        "AND T.DATASOURCE_NUM_ID = S.DATASOURCE_NUM_ID\n",
        "WHEN MATCHED AND S.IND_UPDATE = 'U' THEN UPDATE SET\n",
        "\tT.ID = S.ID,\n",
        "\tT.SKU = S.SKU,\n",
        "\tT.NAME = S.NAME,\n",
        "\tT.PRICE = S.PRICE,\n",
        "\tT.STATUS = S.STATUS,\n",
        "\tT.VISIBILITY = S.VISIBILITY,\n",
        "\tT.WEIGHT = S.WEIGHT,\n",
        "\tT.STOCK_QTY = S.STOCK_QTY,\n",
        "\tT.W_UPDATE_DT = S.W_UPDATE_DT,\n",
        "\tT.PARENT_RX_EBS_PRODUCT_CODE = S.PARENT_RX_EBS_PRODUCT_CODE,\n",
        "\tT.CHANGED_ON_DT = S.CHANGED_ON_DT,\n",
        "\tT.CREATED_ON_DT = S.CREATED_ON_DT,\n",
        "\tT.RX_LINK_TYPE = S.RX_LINK_TYPE,\n",
        "\tT.ETL_PROC_WID = ${ETLProcWID}\n",
        "WHEN NOT MATCHED AND S.IND_UPDATE = 'I' THEN INSERT\n",
        "\t(\n",
        "\tID,\n",
        "\tSKU,\n",
        "\tNAME,\n",
        "\tPRICE,\n",
        "\tSTATUS,\n",
        "\tVISIBILITY,\n",
        "\tWEIGHT,\n",
        "\tSTOCK_QTY,\n",
        "\tINTEGRATION_ID,\n",
        "\tDATASOURCE_NUM_ID,\n",
        "\tW_INSERT_DT,\n",
        "\tW_UPDATE_DT,\n",
        "\tPARENT_RX_EBS_PRODUCT_CODE,\n",
        "\tCHANGED_ON_DT,\n",
        "\tCREATED_ON_DT,\n",
        "\tRX_LINK_TYPE,\n",
        "\tETL_PROC_WID\n",
        "\t)\n",
        "VALUES\n",
        "\t(\n",
        "\tS.ID,\n",
        "\tS.SKU,\n",
        "\tS.NAME,\n",
        "\tS.PRICE,\n",
        "\tS.STATUS,\n",
        "\tS.VISIBILITY,\n",
        "\tS.WEIGHT,\n",
        "\tS.STOCK_QTY,\n",
        "\tS.INTEGRATION_ID,\n",
        "\tS.DATASOURCE_NUM_ID,\n",
        "\tS.W_INSERT_DT,\n",
        "\tS.W_UPDATE_DT,\n",
        "\tS.PARENT_RX_EBS_PRODUCT_CODE,\n",
        "\tS.CHANGED_ON_DT,\n",
        "\tS.CREATED_ON_DT,\n",
        "\tS.RX_LINK_TYPE,\n",
        "\t${ETLProcWID}\n",
        "\t);"
      ],
      "metadata": {
        "language": "sql"
      },
      "outputs": []
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Cleanup"
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {180} (commit - removed), {190}: Drop flow table\n",
        "DROP TABLE IF EXISTS workspace.prxbi_dw.i_wc_badge_product_d_flow;"
      ],
      "metadata": {
        "language": "sql"
      },
      "outputs": []
    },
    {
      "cell_type": "code",
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {220}: Drop staging table\n",
        "DROP TABLE IF EXISTS workspace.prxbi_dw.c_0filter;"
      ],
      "metadata": {
        "language": "sql"
      },
      "outputs": []
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Validation"
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "-- MAGIC %sql\n",
        "SELECT COUNT(*) FROM workspace.prxbi_dw.wc_badge_product_d;"
      ],
      "metadata": {
        "language": "sql"
      },
      "outputs": []
    },
    {
      "cell_type": "code",
      "source": [
        "-- MAGIC %sql\n",
        "SELECT *\n",
        "FROM workspace.prxbi_dw.wc_badge_product_d\n",
        "ORDER BY W_UPDATE_DT DESC\n",
        "LIMIT 100;"
      ],
      "metadata": {
        "language": "sql"
      },
      "outputs": []
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Conversion Notes and Manual Actions Required\n",
        "\n",
        "- **`WC_BADGE_PRODUCT_D` DDL**: The DDL for the target table `WC_BADGE_PRODUCT_D` was not provided in the source. Ensure this table exists in `workspace.prxbi_dw` with the following schema, and `ROW_WID` configured as `GENERATED BY DEFAULT AS IDENTITY`:\n",
        "```sql\n",
        "CREATE TABLE IF NOT EXISTS workspace.prxbi_dw.wc_badge_product_d (\n",
        "    ROW_WID BIGINT GENERATED BY DEFAULT AS IDENTITY,\n",
        "    ID STRING,\n",
        "    SKU STRING,\n",
        "    NAME STRING,\n",
        "    PRICE DECIMAL(25,5),\n",
        "    STATUS BIGINT,\n",
        "    VISIBILITY BIGINT,\n",
        "    WEIGHT BIGINT,\n",
        "    STOCK_QTY BIGINT,\n",
        "    INTEGRATION_ID STRING,\n",
        "    DATASOURCE_NUM_ID STRING,\n",
        "    W_INSERT_DT TIMESTAMP,\n",
        "    W_UPDATE_DT TIMESTAMP,\n",
        "    ETL_PROC_WID BIGINT,\n",
        "    PARENT_RX_EBS_PRODUCT_CODE STRING,\n",
        "    CHANGED_ON_DT TIMESTAMP,\n",
        "    CREATED_ON_DT TIMESTAMP,\n",
        "    RX_LINK_TYPE STRING\n",
        ") USING DELTA;\n",
        "```\n",
        "- **`WC_ETL_PARAMETERS` DDL**: Ensure the `workspace.prxbi_dw.wc_etl_parameters` table exists with `ETL_JOB_TYPE`, `etl_last_extract_time`, `etl_current_extract_time`, and `ROW_WID` columns.\n",
        "- **`DATASOURCE_NUM_ID`**: The value `380` is hardcoded in `SCEN_TASK_NO {110}` for `DATASOURCE_NUM_ID`. If this should be dynamic, adjust the `INSERT` statement to use the `${DATASOURCE_NUM_ID}` widget.\n",
        "- **Error Handling (`E$` tables)**: No `E$` error table logic was present in the provided ODI session text. If error capture is required, it needs to be added manually.\n",
        "- **Optimization**: Consider adding `OPTIMIZE` and `VACUUM` statements for `WC_BADGE_PRODUCT_D` in a separate notebook or as part of a scheduled job if not handled here."
      ]
    }
  ]
}
```